In [ ]:
# Install the PyPharao code and py3Dmol
!pip install rdkit
!pip install git+https://github.com/silicos-it/PyPharao.git
!pip install mols2grid py3Dmol

In [ ]:
# Import all required modules
from tqdm import tqdm_notebook as tqdm

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Draw import IPythonConsole

from IPython.display import display
from IPython.display import SVG

import py3Dmol
from ipywidgets import widgets

import mols2grid
from pypharao import *

from pathlib import Path
import requests

In [ ]:
# Build the pharmacophore query from paracetamol
query_mol = Chem.AddHs(Chem.MolFromSmiles("CC(=O)Nc1ccc(O)cc1"))
AllChem.EmbedMolecule(query_mol)
AllChem.UFFOptimizeMolecule(query_mol)
query = query_pharmacophore_from_molecule(query_mol, name="paracetamol")
print(f"\nQuery {query.get_name()!r} ({len(query)} features):")
for p in query:
    print(f"  {p.type.value:<10} center={p.center}")

In [ ]:
view = py3Dmol.view(
    data=Chem.MolToMolBlock(query_mol),  # Convert the RDKit molecule for py3Dmol
    style={"stick": {}, "sphere": {"scale": 0.3}}
)
view.zoomTo()

In [ ]:
# Generate multiple conformations of paracetamol
NUM_CONFS = 5

print(f"Embedding {NUM_CONFS} conformers for paracetamol")
db = Chem.MolFromSmiles("CC(=O)Nc1ccc(O)cc1")
db = Chem.AddHs(db)
AllChem.EmbedMultipleConfs(db, numConfs=NUM_CONFS)

In [ ]:
# Screen all conformers; keep the best per molecule
searcher = PharmacophoreSearch(query)
print("\nScreening (keep='best', metric='tanimoto'):")
best_hits = searcher.screen([db], conformations="all", keep="best")
sorted_hits = sort_match_results(best_hits, key="tanimoto")
print_match_results(sorted_hits, limit=5)

In [ ]:
# Re-screen with keep='all' (every passing conformer becomes a row)
print("\nScreening (keep='all') — every conformer that satisfies min_matches:")
all_hits = searcher.screen([db], conformations="all", keep="all")
print_match_results(all_hits, limit=5)

In [ ]:
# Persist the best-per-molecule hits as an SDF file
SDF_OUT = Path("./paracetamol_hits.sdf")

ranked = sort_match_results(all_hits, sort="descending", key="tanimoto")
n = write_hits_sdf(ranked, SDF_OUT, pharmacophore=query)
print(f"\nWrote {n} aligned molecules to {SDF_OUT}")

In [ ]:
# Create a molecule from the pharmacophore
mol_from_ph = query.to_mol()

In [ ]:
# Now visualize the top 5 hits from the top_hits_ph_1.sdf file
suppl = Chem.SDMolSupplier(Path("./paracetamol_hits.sdf"))

view = py3Dmol.view()
view.addModel(Chem.MolToMolBlock(mol_from_ph), 'sdf')

counter = 0
for mol in suppl:
    if counter >= 5: break
    else: counter += 1
    if mol is None: continue  # Skip unreadable molecules

    # Process your molecule
    view.addModel(Chem.MolToMolBlock(mol), 'sdf')

view.setStyle({'stick': {}, 'sphere': {'radius': 0.3}})
view.show()
